In [29]:
import sys
from pathlib import Path
import torch
import torch.nn as nn


# Go up two levels: notebook -> pretrain -> project
project_root = Path.cwd().parent.parent

sys.path.append(str(project_root))

from gpt_for_text_generation.src.gpt import *

In [30]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,   
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12, 
    "drop_rate": 0.1,      
    "qkv_bias": False
}

torch.manual_seed(42)
model = GPTModel(GPT_CONFIG_124M)
model.eval()


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded = torch.tensor(encoded).unsqueeze(0)
    return encoded

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())



In [31]:
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(model=model, idx=text_to_token_ids(text=start_context, tokenizer=tokenizer), max_new_tokens=10, context_size=GPT_CONFIG_124M["context_length"])

print("output\n",token_ids_to_text(token_ids=token_ids, tokenizer=tokenizer))



output
 Every effort moves you personalizedebted Saint Karl aquOfficers shapingproblem OPEN Davis


In [32]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]
                    
targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107, 588, 11311]])  #  " really like chocolate"]


with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)
print(probas.shape)
next_idx = torch.argmax(probas, dim=-1, keepdim=True)
print(next_idx)
print("target batch1: ", token_ids_to_text(targets[0], tokenizer))
print("output batch1: ", token_ids_to_text(next_idx[0].flatten(), tokenizer))

torch.Size([2, 3, 50257])
tensor([[[31143],
         [16865],
         [42970]],

        [[17794],
         [34935],
         [  756]]])
target batch1:   effort moves you
output batch1:   mornings recalls Sieg


In [33]:
text_idx = 0
target_probas_1 = probas[text_idx, [0,1,2], targets[text_idx]]
print(target_probas_1)


text_idx = 1
target_probas_2 = probas[text_idx, [0,1,2], targets[text_idx]]
print(target_probas_2)

tensor([6.5647e-06, 9.9088e-06, 1.0423e-05])
tensor([7.0458e-06, 1.2564e-05, 1.8760e-05])


In [34]:
log_probas = torch.log(torch.cat([target_probas_1, target_probas_2]))
print(log_probas)
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor([-11.9338, -11.5221, -11.4715, -11.8631, -11.2847, -10.8838])
tensor(-11.4932)
tensor(11.4932)


In [35]:
#doing the same thing using cross_entropy
logits_flat = logits.flatten(0,1)
targets_flat = targets.flatten()
print(logits_flat.shape)
print(targets_flat.shape)

loss = nn.functional.cross_entropy(logits_flat, targets_flat)
perplexity = torch.exp(loss)
print(perplexity)

torch.Size([6, 50257])
torch.Size([6])
tensor(98043.5391)


In [36]:
with open("../../tokenizer/the-verdict.txt", "r") as f:
    text_data = f.read()


train_ratio = 0.9
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

torch.manual_seed(123)
train_loader = create_dataloader_v1(
    text=train_data, 
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    shuffle=True, 
    drop_last=True,
    num_workers=0
    )

val_loader = create_dataloader_v1(
    text=val_data, 
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    shuffle=True, 
    drop_last=True,
    num_workers=0
    )



In [37]:
print("train loader")
for x, y in train_loader:
    print(x.shape, y.shape)

print("val loader")
for x, y in val_loader:
    print(x.shape, y.shape)

train loader
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
val loader
torch.Size([2, 256]) torch.Size([2, 256])


In [38]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
     
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            total_loss += calc_loss_batch(input_batch, target_batch, model, device)
        else: 
            break
    return total_loss/ num_batches

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)
print("training loss: ", train_loss)
print("validation loss: ", val_loss)

training loss:  tensor(10.9872)
validation loss:  tensor(10.9518)


In [40]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):

    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if (global_step % eval_freq) == 0:
                train_loss, val_loss = eval_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}) "
                      f"train_loss {train_loss:.3f}" 
                      f"val_loss {val_loss:.3f}"
                      )
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen


def eval_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, eval_iter)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model=model, idx=encoded, max_new_tokens=50, context_size=context_size)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [41]:
# torch.manual_seed(123)
# model = GPTModel(GPT_CONFIG_124M)
# model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1) 
# num_epochs = 10
# train_losses, val_losses, tokens_seen = train_model_simple(model, train_loader=train_loader, val_loader=val_loader, optimizer=optimizer, device=device, num_epochs=num_epochs, eval_freq=5, eval_iter=5, start_context="every effort moves you", tokenizer=tokenizer)

In [42]:
# import matplotlib.pyplot as plt
# from matplotlib.ticker import MaxNLocator
# def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
#     fig, ax1 = plt.subplots(figsize=(5, 3))
#     ax1.plot(epochs_seen, train_losses, label="Training loss")
#     ax1.plot(
# epochs_seen, val_losses, linestyle="-.", label="Validation loss"
#     )
#     ax1.set_xlabel("Epochs")
#     ax1.set_ylabel("Loss")
#     ax1.legend(loc="upper right")
#     ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
#     ax2 = ax1.twiny()                  
#     ax2.plot(tokens_seen, train_losses, alpha=0)    
#     ax2.set_xlabel("Tokens seen")
#     fig.tight_layout()
#     plt.show()

In [43]:
# epochs_tensors = torch.linspace(0, num_epochs, len(train_losses))
# plot_losses(epochs_tensors, tokens_seen, train_losses, val_losses)

In [44]:
vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 
inverse_vocab = {v: k for k, v in vocab.items()}

next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

In [45]:
probas = torch.softmax(next_token_logits, dim=-1)
next_token_id = torch.multinomial(probas, num_samples=1).item()
inverse_vocab[next_token_id]

'forward'

In [46]:
def print_sampled_tokens(probas):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item() for _ in range(1000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")
    
print_sampled_tokens(probas)

73 x closer
0 x every
0 x effort
582 x forward
2 x inches
0 x moves
0 x pizza
343 x toward


In [47]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits/temperature
    return torch.softmax(scaled_logits, dim=-1)

softmax_with_temperature(next_token_logits, temperature=0.1)

tensor([1.8530e-10, 3.5189e-26, 2.6890e-38, 9.9099e-01, 5.7569e-23, 4.4220e-37,
        2.9718e-38, 9.0133e-03, 2.8514e-22])

In [48]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, k=top_k)
print(top_logits)
print(top_pos)

tensor([6.7500, 6.2800, 4.5100])
tensor([3, 7, 0])


In [49]:
next_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(-torch.inf), 
    other=next_token_logits
)

In [50]:
tok_proabs = torch.softmax(next_logits, dim=-1)
print(tok_proabs)

tensor([0.0615, 0.0000, 0.0000, 0.5775, 0.0000, 0.0000, 0.0000, 0.3610, 0.0000])


In [51]:
#so lets modify our generate_text_simple function to add tempearature and top-k functionalities
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_value = top_logits[:, -1]
            logits = torch.where(
                condition= logits < min_value,
                input= torch.tensor(-torch.inf).to(device),
                other= logits
            )
        
        if temperature > 0.0:
            logits = logits/temperature
            probas = torch.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probas, num_samples=1)
        else:
            next_idx = torch.argmax(logits, dim=-1, keepdim=True)
        if next_idx == eos_id:
            break
        idx = torch.cat([idx, next_idx], dim=-1)
    return idx


In [52]:
torch.manual_seed(123)
token_ids = generate(model=model,
                     idx=text_to_token_ids("Every effort moves you", tokenizer),
                     max_new_tokens=15,
                     context_size=GPT_CONFIG_124M['context_length'],
                     temperature=1.4,
                     top_k=25
                     )

print("output:\n", token_ids_to_text(token_ids, tokenizer))

output:
 Every effort moves you exteriorwickGraequality Damascus delayed amount thrivingPutting consumed Portal Mari Jones lacksickson


In [53]:
# save the model
torch.save(model.state_dict(), "model.pth")

In [54]:
from torch.serialization import MAP_LOCATION


model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth", map_location=device))
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_feature

In [56]:
# lets also save the optimizer as well as states

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
           }, 
           "model_and_optimizer.pth"
            )

In [57]:
# to load it 
checkpoint = torch.load("model_and_optimizer.pth", map_location=device)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_feature